# Looping Over Data Files

---

[Watch a walk-through of this lesson on YouTube](https://youtu.be/zcg28Qz0ToY)



## Questions:
- How can I efficiently read in many data sets from different files?
- How can I combine data from different files into one pandas DataFrame?

## Learning Objectives:
- Be able to write "globbing" expressions that match sets of files
- Use `glob` to create lists of files
- Write `for` loops to perform operations on many files 
- Write list comprehensions to perform operations on many files
- combine pandas DataFrames

---

## Use a `for` loop to process files given a list of their names

We can use a `for` loop to read in a set of data files, and do a thing for each one. In this case, we'll print the minimum value in each file:

~~~python
import pandas as pd

data_files = ['data/gapminder_gdp_africa.csv', 'data/gapminder_gdp_asia.csv']

for filename in data_files:
    data = pd.read_csv(filename, index_col='country')
    print(filename, data.min())
~~~

In [2]:
import pandas as pd

data_files = ['data/gapminder_gdp_africa.csv', 'data/gapminder_gdp_asia.csv']

for filename in data_files:
    data = pd.read_csv(filename, index_col='country')
    print(filename, data.min())

data/gapminder_gdp_africa.csv gdpPercap_1952    298.846212
gdpPercap_1957    335.997115
gdpPercap_1962    355.203227
gdpPercap_1967    412.977514
gdpPercap_1972    464.099504
gdpPercap_1977    502.319733
gdpPercap_1982    462.211415
gdpPercap_1987    389.876185
gdpPercap_1992    410.896824
gdpPercap_1997    312.188423
gdpPercap_2002    241.165876
gdpPercap_2007    277.551859
dtype: float64
data/gapminder_gdp_asia.csv gdpPercap_1952    331.0
gdpPercap_1957    350.0
gdpPercap_1962    388.0
gdpPercap_1967    349.0
gdpPercap_1972    357.0
gdpPercap_1977    371.0
gdpPercap_1982    424.0
gdpPercap_1987    385.0
gdpPercap_1992    347.0
gdpPercap_1997    415.0
gdpPercap_2002    611.0
gdpPercap_2007    944.0
dtype: float64


## Use [`glob.glob`](https://docs.python.org/3/library/glob.html#glob.glob) to find sets of files whose names match a pattern.

*   In Unix, the term ***globbing*** means matching a set of files with a pattern.
*   The most common patterns are:
    *   `*` meaning match zero or more characters
    *   `?` meaning match exactly one character
*   Python's standard library contains the [`glob`](https://docs.python.org/3/library/glob.html) module to provide pattern matching functionality
*   The [`glob`](https://docs.python.org/3/library/glob.html) module contains a function also called `glob` to match file patterns
*   E.g., `glob.glob('*.txt')` matches all files in the current directory 
    whose names end with `.txt`.
*   Result is a list of strings.

~~~python
import glob
print('all csv files in data directory:', glob.glob('data/*.csv'))

~~~

In [3]:
import glob
print('all csv files in data directory:', glob.glob('data/*.csv'))

all csv files in data directory: ['data\\gapminder_all.csv', 'data\\gapminder_gdp_africa.csv', 'data\\gapminder_gdp_americas.csv', 'data\\gapminder_gdp_asia.csv', 'data\\gapminder_gdp_europe.csv', 'data\\gapminder_gdp_oceania.csv', 'data\\gapminder_life_expectancy_years.csv', 'data\\s1.csv', 'data\\s2.csv', 'data\\s3.csv']


## Use `glob` and `for` to process batches of files.

It's good practice to name your files systematically. As you've learned, Python is very precise about things like capitalization, so if your file names are inconsistent (e.g., `Gapminder_Europe.csv`, `gapminder_americas.csv`, `gapminder_Oceania.csv`), then it is harder to write code with `glob` that works correctly. 

For the Gapminder data, fortunately the file names are quite systematic and consistent (as are the names of the columns inside each file), so we can use the following to read in each one and print the minimum GDP from 1952:

~~~python
for filename in glob.glob('data/gapminder_gdp*.csv'):
    data = pd.read_csv(filename)
    print(filename, data['gdpPercap_1952'].min())
~~~

In [8]:
for filename in glob.glob('data/gapminder_gdp*.csv'):
    data = pd.read_csv(filename)
    print(filename, data['gdpPercap_1952'].min())

data\gapminder_gdp_africa.csv 298.8462121
data\gapminder_gdp_americas.csv 1397.717137
data\gapminder_gdp_asia.csv 331.0
data\gapminder_gdp_europe.csv 973.5331948
data\gapminder_gdp_oceania.csv 10039.59564


## Appending Files to a Single DataFrame

Often we don't just want to open a file and extract a small bit of data (such as the minimum value in examples above). Rather, we might want to open a set of related data files and combine them into one big DataFrame. For example, in psychology and neuroscience most experiments involve multiple participants. For each participant, when we run the experiment we get a data file. To analyze the data across participants, we would want to read in all participants' data files and combined them into one DataFrame.

pandas has a few methods that allow us to combine DataFrames, including:
- [`.concat()`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)
- [`.merge()`](https://pandas.pydata.org/docs/reference/api/pandas.merge.html?highlight=merge#)
- [`.append()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.append.html?highlight=append#pandas.DataFrame.append)
- [`.join()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html#pandas.DataFrame.join)

We will focus here on the first one. `concat` stands for "concatenate" which essentially means combine files by "stacking" them. That is, start with one DataFrame, and add a new data frame to the bottom of it, creating additional rows. In what we'll do here, we assume that all of the data files we're reading have the same columns. For example, in the Gapminder GDP data sets, each file has a column for `country` plus a series of columns for GDP in different years — and the same years are in the columns of all the data sets. 

### Reading in data from multiple experimental participants

Let's say we have data from an experiment in which we ran three human participants (sometimes called "people") on different days. For each participant, we have a data file. The columns in all the files are the same, because the files were generated by a computer program that ran the experiment.

We give the participants anonymized ID codes to protect their privacy, and allow for a simple, systematic naming convention for the files. The first participant's data is saved in a file called `s1.csv`, the second's in `s2.csv`, etc..

We can glob the data folder in which the files are stored, to find all the CSV files whose names start with an `s` followed by a single character, followed by `.csv`. We'll save the result to a list that we can loop through later:

~~~python
filenames = glob.glob('data/s?.csv')
~~~

In [9]:
filenames = glob.glob('data/s?.csv')

Next, we create an empty list that we will store the DataFrames from each participant in. It will end up being a list of DataFrames (remember, lists can contain just about any other Python data type), and once we have read in all the data, we will combine them into one DataFrame. This is a trick that's important to use in pandas. The reason has to do with how pandas combines DataFrames and stores them in memory. In simple terms, each time we concatenate DataFrames, pandas does a lot of internal checking to make sure there are no errors. Doing this checking once, when combining many DataFrames, is far more efficient (and thus faster) than doing it many times. Likewise, when a DataFrame is created, an appropriate amount of memory space is allocated for it on the computer. Each time we append additional data, we have to create a new, bigger block of memory. Allocating new blocks of memory, many times, takes more time than just doing it once.

~~~python
df_list = []
~~~

In [10]:
df_list = []

Finally, use a `for` loop to read the files in. This will cycle through the items in the `filenames` list; each time through the loop, `filename` has the value of the current file name, and we use the list `append()` method to add the data from that file to `df_list`:

~~~python
for f in filenames:
    df_list.append(pd.read_csv(f))
~~~

In [11]:
for f in filenames:
    df_list.append(pd.read_csv(f))

When we view the contents of the list, we see each data set, with its two columns (with headers saying what they are), and commas separating the list entries, as is typical of a list. 

~~~python
df_list
~~~

In [14]:
df_list

[  participantID  trial        RT
 0            s1      1  0.508971
 1            s1      2  0.389858
 2            s1      3  0.404175
 3            s1      4  0.269520
 4            s1      5  0.437765
 5            s1      6  0.368142
 6            s1      7  0.400544
 7            s1      8  0.335198
 8            s1      9  0.341722
 9            s1     10  0.439583,
   participantID  trial        RT
 0            s2      1  0.433094
 1            s2      2  0.392526
 2            s2      3  0.396831
 3            s2      4  0.417988
 4            s2      5  0.371810
 5            s2      6  0.659228
 6            s2      7  0.411051
 7            s2      8  0.409580
 8            s2      9  0.486828
 9            s2     10  0.468912,
   participantID  trial        RT
 0            s3      1  0.322099
 1            s3      2  0.396106
 2            s3      3  0.384297
 3            s3      4  0.364524
 4            s3      5  0.454075
 5            s3      6  0.494156
 6          

## Reading multiple files using list comprehension

While the `for` loop above works fine, there is an alternative way to do this, using [**list comprehension**](https://neuraldatascience.io/3/for-loops.html#list-comprehension). Recall that list comprehensions are basically just a compact version of a `for` loop, but they have some advantages:
- they are *more pythonic*: they only require one line of code, whereas the `for` loop above required two
- they are *more efficient*: list comprehensions actually run faster. This may not be an issue in the small examples here, but can make a big difference when working with real, large data sets

~~~python
df_list = [pd.read_csv(f) for f in filenames]
df_list
~~~

In [16]:
df_list = [pd.read_csv(f) for f in filenames]
df_list

[  participantID  trial        RT
 0            s1      1  0.508971
 1            s1      2  0.389858
 2            s1      3  0.404175
 3            s1      4  0.269520
 4            s1      5  0.437765
 5            s1      6  0.368142
 6            s1      7  0.400544
 7            s1      8  0.335198
 8            s1      9  0.341722
 9            s1     10  0.439583,
   participantID  trial        RT
 0            s2      1  0.433094
 1            s2      2  0.392526
 2            s2      3  0.396831
 3            s2      4  0.417988
 4            s2      5  0.371810
 5            s2      6  0.659228
 6            s2      7  0.411051
 7            s2      8  0.409580
 8            s2      9  0.486828
 9            s2     10  0.468912,
   participantID  trial        RT
 0            s3      1  0.322099
 1            s3      2  0.396106
 2            s3      3  0.384297
 3            s3      4  0.364524
 4            s3      5  0.454075
 5            s3      6  0.494156
 6          

## Combining DataFrames

At this point, we've read each input file in and stored it as a DataFrame, but we have a list of three distinct DataFrames. In most cases, we'll want to combine these in some way. Having built our list of DataFrames through reading a set of files, we can combine them into a single DataFrame using the pandas `.concat()` method:

~~~python
df = pd.concat(df_list)



In [17]:
df = pd.concat(df_list)

Confirm this worked by viewing a random sample of rows
~~~
df.sample(8)
~~~

In [18]:
df.sample(8)

,participantID,trial,RT
2,s2,3,0.396831
4,s1,5,0.437765
5,s3,6,0.494156
0,s3,1,0.322099
8,s3,9,0.340722
1,s3,2,0.396106
7,s1,8,0.335198
3,s2,4,0.417988


## Setting the index column

Recall that row labels in pands are called *indexes*. We can convert any column to an index using the `.set_index()` method. For this data, an appropriate index is the participant ID, which is in the column `Participant`. Note that we need to assign the result of the `.set_index()` operation back to `df` for the change to be stored:

~~~python
df = df.set_index('Participant')
df.sample(8)
~~~

In [21]:
df = df.set_index('participantID')
df.sample(3)

,trial,RT
participantID,,
s1,5,0.437765
s1,10,0.439583
s1,2,0.389858


---
# Exercises
## Determining Matches

Which of these files is *not* matched by the expression `glob.glob('data/*as*.csv')`?

1. `data/gapminder_gdp_africa.csv`
2. `data/gapminder_gdp_americas.csv`
3. `data/gapminder_gdp_asia.csv`

```{admonition} Click the button to reveal the answer
:class: dropdown

1 is not matched. The string `as` occurs in both americ**as** and **as**ia

```

## Globbing files

Fill in the blanks so that the code below does the following: 
- Find all of the CSV files in the data folder that contain GDP data
- Read these files in using a `for` loop
- Concatenate the data files into a single pandas DataFrame
- Print out the first 10 lines of the final combined DataFrame

*Note* that not all the Gapminder data files contain GDP data, but the file names will indicate which ones do. 

In [23]:
import glob
import pandas as pd

data_files = glob.glob('data/*gdp*.csv')

df_list = []

for file in data_files:
    df_list.append(pd.read_csv(file))
    
df = pd.concat(df_list)

df.sample(10)

,country,gdpPercap_1952,gdpPercap_1957,gdpPercap_1962,gdpPercap_1967,gdpPercap_1972,gdpPercap_1977,gdpPercap_1982,gdpPercap_1987,gdpPercap_1992,gdpPercap_1997,gdpPercap_2002,gdpPercap_2007,continent
10,Israel,4086.522128,5385.278451,7105.630706,8393.741404,12786.932230,13306.619210,15367.029200,17122.479860,18051.522540,20896.609240,21905.595140,25523.277100,NaN
17,Netherlands,8941.571858,11276.193440,12790.849560,15363.251360,18794.745670,21209.059200,21399.460460,23651.323610,26790.949610,30246.130630,33724.757780,36797.933320,NaN
0,Algeria,2449.008185,3013.976023,2550.816880,3246.991771,4182.663766,4910.416756,5745.160213,5681.358539,5023.216647,4797.295051,5288.040382,6223.367465,NaN
28,Malawi,369.165080,416.369806,427.901086,495.514781,584.621971,663.223677,632.803921,635.517363,563.200014,692.275810,665.423119,759.349910,NaN
24,Slovenia,4215.041741,5862.276629,7402.303395,9405.489397,12383.486200,15277.030170,17866.721750,18678.534920,14214.716810,17161.107350,20660.019360,25768.257590,NaN
41,Sierra Leone,879.787736,1004.484437,1116.639877,1206.043465,1353.759762,1348.285159,1465.010784,1294.447788,1068.696278,574.648158,699.489713,862.540756,NaN
3,Bosnia and Herzegovina,973.533195,1353.989176,1709.683679,2172.352423,2860.169750,3528.481305,4126.613157,4314.114757,2546.781445,4766.355904,6018.975239,7446.298803,NaN
31,Mauritius,1967.955707,2034.037981,2529.067487,2475.387562,2575.484158,3710.982963,3688.037739,4783.586903,6058.253846,7425.705295,9021.815894,10956.991120,NaN
11,Congo Rep.,2125.621418,2315.056572,2464.783157,2677.939642,3213.152683,3259.178978,4879.507522,4201.194937,4016.239529,3484.164376,3484.061970,3632.557798,NaN
10,El Salvador,3048.302900,3421.523218,3776.803627,4358.595393,4520.246008,5138.922374,4098.344175,4140.442097,4444.231700,5154.825496,5351.568666,5728.353514,Americas


```{admonition} Click the button to reveal!
:class: dropdown

~~~python
import glob
import pandas as pd

data_files = glob.glob('data/*gdp*.csv')

df_list = []

for f in data_files:
    df_list.append(pd.read_csv(f))
    
df = pd.concat(df_list)

df.head(10)
~~~
```

### List comprehension

Now rewrite the code above to use list comprehension rather than a `for` loop, and only *two* lines of code total (excluding the `import` commands and viewing the first 10 lines of the result). 

In [33]:
df_list = [pd.read_csv(f) for f in glob.glob('data/*gdp*.csv')]
df = pd.concat(df_list)
df.sample(10)

,country,gdpPercap_1952,gdpPercap_1957,gdpPercap_1962,gdpPercap_1967,gdpPercap_1972,gdpPercap_1977,gdpPercap_1982,gdpPercap_1987,gdpPercap_1992,gdpPercap_1997,gdpPercap_2002,gdpPercap_2007,continent
3,Botswana,851.241141,918.232535,983.653976,1214.709294,2263.611114,3214.857818,4551.142150,6205.883850,7954.111645,8647.142313,11003.605080,12569.851770,NaN
9,Iraq,4129.766056,6229.333562,8341.737815,8931.459811,9576.037596,14688.235070,14517.907110,11643.572680,3745.640687,3076.239795,4390.717312,4471.061906,NaN
16,Nicaragua,3112.363948,3457.415947,3634.364406,4643.393534,4688.593267,5486.371089,3470.338156,2955.984375,2170.151724,2253.023004,2474.548819,2749.320965,Americas
10,Germany,7144.114393,10187.826650,12902.462910,14745.625610,18016.180270,20512.921230,22031.532740,24639.185660,26505.303170,27788.884160,30035.801980,32170.374420,NaN
11,Guatemala,2428.237769,2617.155967,2750.364446,3242.531147,4031.408271,4879.992748,4820.494790,4246.485974,4439.450840,4684.313807,4858.347495,5186.050003,Americas
10,Congo Dem. Rep.,780.542326,905.860230,896.314634,861.593242,904.896068,795.757282,673.747818,672.774812,457.719181,312.188423,241.165876,277.551859,NaN
23,Uruguay,5716.766744,6150.772969,5603.357717,5444.619620,5703.408898,6504.339663,6920.223051,7452.398969,8137.004775,9230.240708,7727.002004,10611.462990,Americas
20,Portugal,3068.319867,3774.571743,4727.954889,6361.517993,9022.247417,10172.485720,11753.842910,13039.308760,16207.266630,17641.031560,19970.907870,20509.647770,NaN
47,Togo,859.808657,925.908320,1067.534810,1477.596760,1649.660188,1532.776998,1344.577953,1202.201361,1034.298904,982.286924,886.220576,882.969944,NaN
21,Oman,1828.230307,2242.746551,2924.638113,4720.942687,10618.038550,11848.343920,12954.791010,18115.223130,18616.706910,19702.055810,19774.836870,22316.192870,NaN


For an even bigger challenge, see if you can reduce the code to a single line!

In [37]:
df = pd.concat([pd.read_csv(f) for f in glob.glob('data/*gdp*.csv')])
df.sample(10)

,country,gdpPercap_1952,gdpPercap_1957,gdpPercap_1962,gdpPercap_1967,gdpPercap_1972,gdpPercap_1977,gdpPercap_1982,gdpPercap_1987,gdpPercap_1992,gdpPercap_1997,gdpPercap_2002,gdpPercap_2007,continent
42,Somalia,1135.749842,1258.147413,1369.488336,1284.733180,1254.576127,1450.992513,1176.807031,1093.244963,926.960296,930.596428,882.081822,926.141068,NaN
19,Peru,3758.523437,4245.256698,4957.037982,5788.093330,5937.827283,6281.290855,6434.501797,6360.943444,4446.380924,5838.347657,5909.020073,7408.905561,Americas
30,Mauritania,743.115910,846.120261,1055.896036,1421.145193,1586.851781,1497.492223,1481.150189,1421.603576,1361.369784,1483.136136,1579.019543,1803.151496,NaN
38,Rwanda,493.323875,540.289398,597.473073,510.963714,590.580664,670.080601,881.570647,847.991217,737.068595,589.944505,785.653765,863.088464,NaN
29,United Kingdom,9979.508487,11283.177950,12477.177070,14142.850890,15895.116410,17428.748460,18232.424520,21664.787670,22705.092540,26074.531360,29478.999190,33203.261280,NaN
27,Switzerland,14734.232750,17909.489730,20431.092700,22966.144320,27195.113040,26982.290520,28397.715120,30281.704590,31871.530300,32135.323010,34480.957710,37506.419070,NaN
6,Cameroon,1172.667655,1313.048099,1399.607441,1508.453148,1684.146528,1783.432873,2367.983282,2602.664206,1793.163278,1694.337469,1934.011449,2042.095240,NaN
15,Italy,4931.404155,6248.656232,8243.582340,10022.401310,12269.273780,14255.984750,16537.483500,19207.234820,22013.644860,24675.024460,27968.098170,28569.719700,NaN
23,Uruguay,5716.766744,6150.772969,5603.357717,5444.619620,5703.408898,6504.339663,6920.223051,7452.398969,8137.004775,9230.240708,7727.002004,10611.462990,Americas
25,Spain,3834.034742,4564.802410,5693.843879,7993.512294,10638.751310,13236.921170,13926.169970,15764.983130,18603.064520,20445.298960,24835.471660,28821.063700,NaN


```{admonition} Click the button to reveal!
:class: dropdown

Done in two lines of code:
~~~python
df_list = [pd.read_csv(f) for f in glob.glob('data/*gdp*.csv')]
    
df = pd.concat(df_list)

df.head(10)
~~~

Done in one line of code: 

~~~python
df = pd.concat([pd.read_csv(f) for f in glob.glob('data/*gdp*.csv')])
    
df.head(10)
~~~

```

## Summary of Key Points:
- Use a `for` loop to process files given a list of their names
- Use `glob.glob` to find sets of files whose names match a pattern
- List comprehension can replace a `for` loop, resulting in more compact and efficient code
- Naming your files in a consistent manner is just as important in data science, as writing the code to read them
- When you want to combine multiple files into one pandas DataFrame, read each one in to a list of DataFrames, then run `pd.concat()` only once

---
This lesson is adapted from the [Software Carpentry](https://software-carpentry.org/lessons/) [Plotting and Programming in Python](http://swcarpentry.github.io/python-novice-gapminder/) workshop. 